# ARU-Net Demo

Minimal end-to-end demonstration of the ARU-Net image reconstruction pipeline: build the model, run a forward pass, compute the composite loss, and (if data is available) train for a few epochs and visualize reconstructions.

This notebook is a **clean demo**, not a copy of the original research notebook (`4_AttentionResidualUNet.ipynb`). For the fully documented, modular implementation, see `src/`. For the audit of thesis-vs-code discrepancies, see the top-level `README.md` and `docs/methodology.md`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from src.model import AttentionResidualUNet, count_parameters
from src.losses import CombinedLoss
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()
print('Device:', device)

## 1. Build the model

In [ ]:
model = AttentionResidualUNet().to(device)
print(f'Parameters: {count_parameters(model):,}')

## 2. Sanity-check a forward pass

In [ ]:
dummy_input = torch.rand(2, 3, 256, 256).to(device)
with torch.no_grad():
    output = model(dummy_input)
print('Input shape :', tuple(dummy_input.shape))
print('Output shape:', tuple(output.shape))

## 3. Compute the composite loss on random tensors

Set `use_vgg=False` here if you don't have internet access to download ImageNet-pretrained VGG16 weights (e.g. in an offline sandbox).

In [ ]:
target = torch.rand(2, 3, 256, 256).to(device)
criterion = CombinedLoss(device=device, use_vgg=True)
loss = criterion(output, target)
print('Loss:', loss.item())

## 4. Train on real data (optional)

Requires `data/clean/` and `data/distorted/<dataset_name>/` to be populated -- see `data/README.md`. Uncomment to run.

In [ ]:
# from torch.utils.data import DataLoader
# from configs.config import get_config
# from src.dataset import ImagePairDataset, split_dataset
# from src.train import train
#
# cfg = get_config()
# cfg.paths.dataset_name = 'Noise_Multiply_Strong'
# cfg.train.epochs = 5  # short demo run
#
# full_dataset = ImagePairDataset(
#     distorted_dir=str(cfg.paths.distorted_dir),
#     clean_dir=str(cfg.paths.clean_dir),
#     dist_suffix=cfg.dataset.dist_suffix,
#     clean_suffix=cfg.dataset.clean_suffix,
# )
# train_dataset, val_dataset, test_dataset = split_dataset(full_dataset, cfg)
# train_loader = DataLoader(train_dataset, batch_size=cfg.train.batch_size_train, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=cfg.train.batch_size_eval, shuffle=False)
#
# train_losses = train(model, train_loader, val_loader, device, epochs=cfg.train.epochs,
#                       save_dir='../results/reconstructions/demo', patience=cfg.train.early_stopping_patience)

## 5. Visualize sample reconstructions (optional)

Requires a trained checkpoint and a populated test set.

In [ ]:
# from src.evaluate import visualize_sample_results
# visualize_sample_results(model, test_loader, device, num_samples=3, save_dir='../results/figures/demo')